In [11]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [12]:
use_cols = [
    "item_id",
    "store_id",
    "date",
    "sales",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1"
]

CHUNKSIZE = 2_000_000

chunks = pd.read_csv(
    "../data/interim/sales_long.csv",
    usecols=use_cols,
    parse_dates=["date"],
    chunksize=CHUNKSIZE
)



In [13]:
def create_lag_features(df):
    df = df.sort_values("date")
    
    # Lag features
    for lag in [7, 14, 28]:
        df[f"lag_{lag}"] = df["sales"].shift(lag)
    
    # Rolling features
    for window in [7, 14, 28]:
        df[f"rmean_{window}"] = (
            df["sales"]
            .shift(1)
            .rolling(window)
            .mean()
        )
        
    return df


In [14]:
history = {}


In [15]:
first_write = True

total_rows = 0
num_chunks = 0
num_cols = None


for chunk in chunks:
    processed = []

    for item_id, g in chunk.groupby("item_id"):
        # prepend history if exists
        if item_id in history:
            g = pd.concat([history[item_id], g])

        # create features
        g = create_lag_features(g)

        # update history
        history[item_id] = g.tail(28)

        # drop overlap rows
        g = g.iloc[28:]

        processed.append(g)

    chunk_fe = pd.concat(processed, ignore_index=True)

    # add event flag
    chunk_fe["is_event"] = chunk_fe["event_name_1"].notna().astype(int)

    # remove NaNs
    chunk_fe = chunk_fe.dropna()

    total_rows += chunk_fe.shape[0]
    num_cols = chunk_fe.shape[1]
    num_chunks += 1


    # write incrementally
    chunk_fe.to_csv(
        "../data/processed/train_fe.csv",
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )

    first_write = False


print(f"Total rows: {total_rows}")
print(f"Columns: {num_cols}")
print(f"Chunks processed: {num_chunks}")

Total rows: 4695460
Columns: 17
Chunks processed: 30


## Feature Engineering Summary

- Feature engineering was performed on the **entire M5 dataset across all stores and items** using a **chunk-based processing pipeline** to ensure scalability and memory efficiency.
- Lag features (7, 14, and 28 days) were created at the **item level** to capture short- and medium-term temporal dependencies in sales.
- Rolling mean features (7, 14, and 28 days) were computed using **shifted sales values** to smooth highly intermittent demand patterns while preventing information leakage.
- Calendar-based features (weekday, month, and year) were retained to capture recurring temporal effects.
- A binary event indicator (`is_event`) was created based on the presence of calendar events to allow the model to learn heterogeneous event impacts.
- Feature engineering was applied **group-wise and time-ordered per item**, preserving temporal continuity even when processing the data in chunks.
- Rows with insufficient historical context (introduced by lag and rolling window requirements) were removed to ensure valid feature values and avoid leakage from future observations.

These engineered features constitute the primary input for downstream forecasting models and baseline evaluations.
